# DACKAR — Outage Activity Analysis Pipeline Demo

**What is this?**  DACKAR is an AI-assisted decision-support system for nuclear plant outage management.
When an unexpected activity is discovered during an outage, DACKAR automatically analyses its history,
temporal context, and schedule impact, then recommends the safest and most efficient course of action.

**Two demo scenarios** exercise every stage of the pipeline and show two opposite outcomes:

| # | Scenario | Emergence Type | CP Impact | Regulatory Constraint | Expected Decision |
|---|----------|---------------|-----------|----------------------|-------------------|
| 1 | RCP Train-A Mechanical Seal Leak | `regulatory_driven` | CRITICAL — 48 h drag | TS 3.4.6 (defer prohibited) | **🔴 ESCALATE** |
| 2 | Snubber Inspection Scope Expansion | `scope_expansion` | NON-CRITICAL — 0 h drag | None | **🟢 PROCEED** |

---

**Stage execution summary:**

| Stage | Name | Mode | Description |
|-------|------|------|-------------|
| A | Activity Intake & Classification | Live | Classifies the activity, assesses data quality |
| B | KG Timeline Builder | Live (stub backend) | Retrieves prior events from the Knowledge Graph |
| C | Temporal Event Chain | Live | Allen interval algebra — was this event caused by something earlier? |
| D | Historical Analog Retriever | Live (stub backend) | Finds similar past activities; fits a duration distribution |
| E | Schedule Impact Assessment | Stub | Computes CP float impact (requires LOGOS CPM engine in production) |
| F | Insertion Option Generator | Live | Generates and risk-scores all viable insertion options |
| G | Recommendation Synthesiser | Live | Produces the final decision + evidence chain |

> **Stub backends** replace Neo4j (KG) and a vector embedding server (retrieval) with
> in-memory fixtures — so the demo runs anywhere with no external services.


## 0 · Setup

This cell imports Python libraries and the two demo scenario definitions.
It also configures the Matplotlib plot style used throughout the notebook.

**For developers:** `demo_scenarios.py` in this folder contains all stub backends
(`_StubKGDriver`, `_StubRetrievalIndex`) and the two pre-built scenario dictionaries.
No external services (Neo4j, embedding server, LOGOS) are needed.


In [ ]:
import sys
import warnings
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.ticker import MaxNLocator
import numpy as np

# Suppress noisy library warnings and debug log spam during the demo
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

# Add the outage package and demo folder to sys.path.
# Notebook lives two levels below outage/:
#   demos/unexpected_act_workflow_1/ → demos/ → outage/
DEMO_DIR   = Path().resolve()              # .../demos/unexpected_act_workflow_1/
OUTAGE_DIR = DEMO_DIR.parent.parent        # .../outage/
if str(OUTAGE_DIR) not in sys.path:
    sys.path.insert(0, str(OUTAGE_DIR))
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

# Import the two pre-built scenario dicts and the orchestrator function
from demo_scenarios import run_pipeline, SCENARIO_RCP_SEAL, SCENARIO_SNUBBER_EXT

# ── Plot style ────────────────────────────────────────────────────────────────
# These settings apply globally to every matplotlib figure in the notebook
plt.rcParams.update({
    'figure.dpi': 110,              # crisp rendering for screen and PDF
    'axes.spines.top': False,       # remove top/right borders for a clean look
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

# Shared colour palette — keeps all plots visually consistent.
# Keys map to decision outcomes (escalate/proceed/defer) and confidence tiers.
PALETTE = {
    'escalate':       '#D7263D',   # red
    'proceed':        '#06A77D',   # teal-green
    'defer':          '#F4A261',   # amber
    'monitor':        '#A8DADC',   # light blue
    'blocked':        '#E76F51',   # orange-red (infeasible but exists)
    'infeasible':     '#CCCCCC',   # light grey
    'data_supported': '#264653',   # dark teal  (≥5 analogues)
    'sme_informed':   '#2A9D8F',   # medium teal (1–4 analogues)
    'low_confidence': '#E9C46A',   # gold        (0 analogues)
    'stage_live':     '#264653',   # dark teal   (stage runs real logic)
    'stage_stub':     '#94A3B8',   # slate grey  (stage uses pre-built data)
    'stage_text':     '#FFFFFF',   # white text on stage tiles
}


# Reusable plot functions — imported from demo_plots.py so the
# notebook stays clean and a future GUI can reuse the same functions.
from demo_plots import (
    draw_pipeline_architecture, plot_stage_a_summary,
    plot_allen_timeline, plot_analog_distribution, plot_schedule_impact,
    plot_option_risk_scores, plot_recommendation_card,
    plot_scenario_comparison, plot_evidence_chain,
)
from demo_scenarios import SCENARIO_UNKNOWN_COMPONENT

print('Setup complete.')


## 1 · Run the Pipeline

The cell below runs both scenarios end-to-end through the DACKAR pipeline.
Each call to `run_pipeline()` executes stages A → B → C → D → E → F → G in sequence
and returns a dictionary containing every stage's output artifact.

**For managers:** This is the single entry point — one function call per scenario.
The pipeline typically completes in under 200 ms (stub backends; production with
Neo4j and embedding server will be in the 1–3 s range per scenario).


In [ ]:
import time

# --- Run Scenario 1: RCP Seal Leak (expected: ESCALATE) ---
t0 = time.perf_counter()
r1 = run_pipeline(SCENARIO_RCP_SEAL)       # dict with keys: scenario_label, run_id,
t1 = time.perf_counter()                   # intake, timeline, temporal, analogs,
                                           # schedule, options, recommendation

# --- Run Scenario 2: Snubber Scope Expansion (expected: PROCEED) ---
t2 = time.perf_counter()
r2 = run_pipeline(SCENARIO_SNUBBER_EXT)
t3 = time.perf_counter()

# Print a quick summary — scenario_label already contains the scenario name,
# so we do NOT add "Scenario N —" prefix here to avoid duplication.
print(r1['scenario_label'])
print(f"  Decision : {r1['recommendation']['decision_status']}")
print(f"  Runtime  : {(t1-t0)*1000:.0f} ms")
print()
print(r2['scenario_label'])
print(f"  Decision : {r2['recommendation']['decision_status']}")
print(f"  Runtime  : {(t3-t2)*1000:.0f} ms")

# --- Run Scenario 3: Unknown Component, No Prior History (expected: MONITOR) ---
t2 = time.perf_counter()
r3 = run_pipeline(SCENARIO_UNKNOWN_COMPONENT)
r3['scenario_label'] = SCENARIO_UNKNOWN_COMPONENT['label']
print(f'Scenario 3 done in {time.perf_counter()-t2:.2f}s  '  
      f'— {r3["recommendation"]["decision_status"]}')


## 2 · Pipeline Architecture

The diagram below shows how the seven stages are connected.
Stages in **dark teal** run real production logic; **grey** stages use pre-built stub data.

Arrows show data flow: each stage's output JSON becomes the next stage's input.
Stages B and D run concurrently (they are independent of each other) in the
production orchestrator, though in this demo they run sequentially for simplicity.


In [ ]:
draw_pipeline_architecture()
plt.show()


---
## 3 · Scenario 1 — RCP Train-A Seal Leak  🔴 ESCALATE

**What happened?** During the outage, a mechanical seal leak was detected on RCP Train-A.
This is a **safety-related** component covered by Technical Specification 3.4.6,
which **prohibits deferral** — the activity must be addressed, but inserting it now
conflicts with crew availability and adds 48 hours of critical-path drag.

**Why ESCALATE?** Every insertion option is either infeasible or contradicted by
regulatory constraints. The pipeline correctly flags this for senior engineering
and plant management review rather than making an autonomous decision.

The cells below walk through each stage's contribution to this conclusion.

> **Outage Manager** — *Should I stop other work and fix this now, or can it wait?*  
> **Data Scientist** — *What entity types, regulatory drivers, and analog count drove the ESCALATE decision?*


### 3.1 · Stage A — Activity Intake & Classification

**What Stage A does:**
Stage A is the pipeline entry point. It takes the raw activity description, assigns a
unique activity ID, classifies the *emergence type* (e.g., `regulatory_driven`,
`scope_expansion`, `equipment_failure`), scores data quality, and flags whether
regulatory constraints are present.

**Key outputs to look for:**
- `emergence_type` and its confidence score
- `data_quality_score` (0–1): how complete and reliable the input data is
- `has_regulatory_constraint`: if True, Stage F will be more restrictive


In [ ]:
# Retrieve the Stage A output artifact for Scenario 1
intake = r1['intake']

print('=== STAGE A — Activity Intake ===')
print(f"  Activity ID          : {intake['activity_id']}")

# emergence_type: how this activity entered the outage scope
# confidence: ML classifier score (1.0 = certain, <0.7 = uncertain)
print(f"  Emergence type       : {intake['emergence_type']}  (confidence {intake['emergence_type_confidence']:.0%})")

# has_regulatory_constraint: True means Stage F cannot mark deferral as feasible
print(f"  Has regulatory const : {intake['has_regulatory_constraint']}")

# data_quality_score: affects confidence tier propagated to Stages D, F, and G
print(f"  Data quality score   : {intake['data_quality_score']:.2f}")


In [ ]:
fig, axes = plot_stage_a_summary(r1['intake'],
                                  title='Stage A — RCP Seal Scenario')
plt.show()


### 3.2 · Stage C — Temporal Event Chain (Allen Interval Algebra)

**What Stage C does:**
Stage C asks: *"Did anything happen before this activity that could explain it?"*

It retrieves the component's event history from the Knowledge Graph (Stage B) and
classifies each historical event using **Allen interval algebra** — a mathematical
framework for describing how two time intervals relate to each other:

| Allen Relation | Meaning | Causal Score |
|---------------|---------|-------------|
| OVERLAPS | Prior event was still active when this activity began | 0.90 |
| CONTAINS | Long-running event that spans the entire activity window | 0.85 |
| PRECEDES | Prior event ended before this activity started (classic lead-time) | 0.75 |
| SIMULTANEOUS | Concurrent — possible common-cause scenario | 0.50 |
| DURING | Started *after* the activity — likely a symptom, not a cause | 0.30 |
| FOLLOWS | Temporal contradiction — flagged for analyst review | 0.10 |

**Key output to look for:** `causal_posture` in the summary
(`supported` / `partial` / `contradicted` / `insufficient_data`)


In [ ]:
fig, ax = plot_allen_timeline(r1['temporal'],
                               title='Stage C — RCP Seal Scenario (Allen Relations)')
plt.show()


### 3.3 · Stage D — Historical Analog Distribution

**What Stage D does:**
Stage D searches the historical database for past activities that are similar
to the current one ("analogues") and fits a statistical duration distribution
to their actual completion times.

This answers: *"How long does this type of work typically take, based on history?"*

**Confidence tiers** (driven by analogue count):

| Tier | Analogues Found | Meaning |
|------|----------------|---------|
| `data_supported` | ≥ 5 | Strong historical evidence; high confidence |
| `sme_informed` | 1–4 | Some history; SME judgment still needed |
| `low_confidence` | 0 | No history found; treat estimates cautiously |

**Key outputs to look for:** `confidence_tier`, `p50_hours` (median), `p80_hours` (planning estimate)


In [ ]:
fig, axes = plot_analog_distribution(r1['analogs'],
                                       title='Stage D — RCP Seal Analog Duration Distribution')
plt.show()
print('Retrieval summary:', r1['analogs'].get('retrieval_summary', {}))


### 3.4 · Stage E — Schedule Impact (Float Analysis)

**What Stage E does:**
Stage E quantifies the schedule impact of inserting the new activity into the
outage critical path. It uses a CPM (Critical Path Method) engine to compute:

- **Float**: how much scheduling slack exists at the insertion point
- **CP drag**: how many hours the activity would add to the total outage duration
- **Remaining float after insertion**: negative = critical path extended

> **Note:** In this demo, Stage E output is pre-built (stub) because the LOGOS
> CPM scheduling engine is not available in the demo environment.
> In production, Stage E calls the LOGOS API to compute these values in real-time.

**Key outputs to look for:** `remaining_float_after_hours` (negative = CP extension),
`cp_drag_hours` (directly added to outage end date)


In [ ]:
fig, axes = plot_schedule_impact(r1['schedule'],
                                  title='Stage E — RCP Seal Schedule Impact')
plt.show()

sch = r1['schedule']
ip = sch.get('insertion_point', {})
print(f"Insertion point : {ip.get('task_name', '?')} (task {ip.get('task_id', '?')})")
displaced = sch.get('displaced_tasks', [])
reg_displaced = sum(1 for t in displaced if t.get('has_regulatory_constraint'))
print(f"Displaced tasks : {len(displaced)}  (regulatory: {reg_displaced})")
for c in sch.get('resource_conflicts', []):
    print(f"  [{c['conflict_type']:25s}] skill={c.get('skill_required')}")


### 3.5 · Stage F — Insertion Option Risk Scoring

**What Stage F does:**
Stage F generates every viable way to handle the new activity and scores
each option's risk using a multi-factor formula:

```
risk = 0.40 × cp_impact  +  0.30 × (1 − confidence)  +  0.20 × resource_score  +  0.10 × urgency
```

**Option types considered:**
| Option | Description |
|--------|-------------|
| `insert_now` | Insert immediately at the best available slot |
| `parallel` | Run concurrently with a non-critical task |
| `defer` | Defer to a later outage (only if regulatory permits) |
| `contingency_buffer` | Use schedule reserve time |
| `escalate` | Escalate to management (triggered when CP drag > 24 h) |

Options can be **infeasible** (crew conflict, critical task displacement) or
**regulatory-cleared = False** (TS constraint prohibits deferral).
The option with the **lowest risk score** among feasible, cleared options wins.

**Key output to look for:** `recommended_option_id` — the winning option


In [ ]:
fig, ax = plot_option_risk_scores(r1['options'],
                                   title='Stage F — RCP Seal Option Risk Scores')
plt.show()

sm = r1['options'].get('ranking_summary', {})
print(f"Options: generated={sm.get('options_generated')} "
      f"feasible={sm.get('feasible_count')} "
      f"cleared={sm.get('regulatory_cleared_count')} "
      f"blocked={sm.get('regulatory_blocked_count')} "
      f"infeasible={sm.get('infeasible_count')} "
      f"best_score={sm.get('best_risk_score')}")


### 3.6 · Stage G — Recommendation Card

**What Stage G does:**
Stage G is the final synthesis stage. It takes all upstream outputs and produces:

1. A **decision status**: ESCALATE / PROCEED / DEFER / MONITOR / INCONCLUSIVE
2. An **executive summary** with the primary conclusion in plain language
3. An **evidence chain**: every data point that informed the decision (traceable)
4. An **analyst review** flag: if True, a human must review before action is taken

**For managers:** The recommendation card is the primary deliverable of the pipeline.
Everything above it is the computational evidence trail.


In [ ]:
def print_recommendation_card(result):
    """Print a formatted text recommendation card for one scenario result.

    Uses emoji status icons instead of ANSI terminal colour codes so the
    output renders correctly in Jupyter notebooks and exported HTML/PDF.
    """
    rec   = result['recommendation']
    summ  = rec.get('executive_summary', {})
    prim  = rec.get('primary_recommendation', {})
    hist  = rec.get('history_summary', {})
    sched = rec.get('schedule_summary', {})
    rev   = rec.get('analyst_review', {})
    flags = rec.get('attention_flags', [])
    reg   = rec.get('regulatory_flags', [])

    status = rec['decision_status']

    # Emoji icons render in Jupyter; ANSI escape codes do not
    STATUS_ICONS = {
        'ESCALATE':     '🔴',
        'PROCEED':      '🟢',
        'DEFER':        '🟡',
        'MONITOR':      '🔵',
        'INCONCLUSIVE': '⚪',
    }
    icon = STATUS_ICONS.get(status, '❓')

    width = 72
    bar   = '═' * width
    print(f'\n{bar}')
    print(f'  DACKAR RECOMMENDATION  —  {result["scenario_label"]}')
    print(bar)
    # Show icon alongside decision status for at-a-glance reading
    print(f'  DECISION          : {icon} {status}')
    print(f'  Confidence tier   : {summ.get("confidence_tier", "?")}')
    print(f'  Analyst review    : {rev.get("required", False)}')
    if rev.get('required') and rev.get('reason'):
        print(f'  Review reason     : {rev["reason"]}')
    print()
    print(f'  Primary conclusion:')
    conclusion = summ.get('primary_conclusion', 'N/A')
    import textwrap
    for line in textwrap.wrap(conclusion, width=66):
        print(f'    {line}')
    print()
    if prim:
        print(f'  Recommended option: {prim.get("option_type", "?")}')
        print(f'  CP impact (hours) : {prim.get("cp_impact_hours", 0):.1f} h')
        rationale = prim.get('rationale', '')
        for line in textwrap.wrap(rationale, width=66):
            print(f'    {line}')
    print()
    if flags:
        print('  Attention flags:')
        for f in flags:
            print(f'    ⚑ {f}')
        print()
    if reg:
        print(f'  Regulatory constraints ({len(reg)}):')
        for d in reg:
            print(f'    [{d["driver_type"]:35s}] defer_prohibited={d["defer_prohibited"]}')
        print()
    if reg_warn := summ.get('regulatory_warning'):
        print(f'  Regulatory warning : {reg_warn}')
    print(f'  Analog count      : {hist.get("analog_count", 0)} events across '
          f'{hist.get("outages_represented", 0)} outages')
    print(f'  Median duration   : {hist.get("median_actual_hours", "?")}')
    print()
    print(f'  Evidence chain    : {len(rec.get("evidence_chain", []))} items')
    for ev in rec.get('evidence_chain', [])[:4]:
        print(f'    [{ev.get("source_type","?")}] {ev.get("snippet","")[:60]}')
    print(bar)

# Print the card for Scenario 1 (RCP Seal — expected ESCALATE)
print_recommendation_card(r1)


In [ ]:
fig, ax = plot_recommendation_card(r1)
plt.show()


---
## 4 · Scenario 2 — Snubber Scope Expansion  🟢 PROCEED

**What happened?** While performing routine snubber inspections, engineers identified
additional snubbers requiring evaluation — a scope expansion common in aging plants.

**Why PROCEED?** This activity is:
- **Non-safety-related** (no TS constraint, deferral is technically allowed)
- **No critical-path impact** (0 h CP drag, 28 h of remaining float)
- **Well-supported by history** (5 analogous past activities → `data_supported` tier)

The pipeline recommends immediate insertion with high confidence.

Compare the cells below with Scenario 1 to see how each stage produces a
completely different evidence trail and conclusion from similar inputs.

> **Outage Manager** — *This is non-critical; the question is how to sequence the work without creating conflicts.*  
> **Data Scientist** — *Five analogues push us to data_supported tier. Note the option scoring without regulatory constraints.*


In [ ]:
# Retrieve and display Stage A output for Scenario 2 (Snubber)
print('=== STAGE A — Snubber Scenario ===')
intake2 = r2['intake']

# Compare with Scenario 1: different emergence_type, no regulatory constraint
print(f"  Emergence type   : {intake2['emergence_type']}  (confidence {intake2['emergence_type_confidence']:.0%})")
print(f"  Has regulatory   : {intake2['has_regulatory_constraint']}")  # False — no TS constraint
print(f"  Data quality     : {intake2['data_quality_score']:.2f}")

# Reuse the same plot function defined for Scenario 1
plot_stage_a_summary(intake2, title=f"Stage A — {r2['scenario_label']}")


In [ ]:
# Stage C for Scenario 2 — Allen relations for snubber component history
# Expect: fewer/weaker prior events than the safety-critical RCP pump
plot_allen_timeline(r2['temporal'], title='Stage C — Snubber Scenario (Allen Relations)')


In [ ]:
# Stage D for Scenario 2 — duration distribution from 5 snubber analogue activities
# With 5 analogues, confidence_tier = 'data_supported' (threshold is ≥5)
# This is key: a higher confidence tier means insert_now beats defer on risk score
plot_analog_distribution(r2['analogs'], title='Stage D — Snubber Analog Duration Distribution')


In [ ]:
# Stage E for Scenario 2 — non-critical activity, zero CP drag
# remaining_float_after_hours = +28.0 (plenty of slack after insertion)
# Compare with Scenario 1 where remaining_float_after_hours = -44.0
plot_schedule_impact(r2['schedule'], title='Stage E — Snubber Schedule Impact (Non-Critical)')


In [ ]:
# Stage F for Scenario 2 — all options feasible, no regulatory constraint
# insert_now wins because: data_supported confidence (0.85) → low risk score
# defer is available but has higher risk due to urgency score
plot_option_risk_scores(r2['options'], title='Stage F — Snubber Option Risk Scores')


In [ ]:
# Stage G for Scenario 2 — print and plot the PROCEED recommendation card
# Note the 🟢 icon (vs 🔴 for Scenario 1) — the same pipeline, opposite conclusion
print_recommendation_card(r2)
plot_recommendation_card(r2)


---
## 4.3 · Scenario 3 — Unknown Component, No Prior History  🔵 MONITOR

**What happened?** A component tag (ACT-UNK-003) was flagged during walkdown. No condition reports or work orders exist for this tag in the plant database. The activity is safety-related, but no Technical Specification clause applies.

**Why MONITOR?** Zero historical analogs → `low_confidence` tier. No causal chain from Stage C (empty KG). Non-critical schedule impact (48 h float remaining). Because the activity is `safety_related=True`, deferral to post-outage is blocked. The pipeline cannot make a defensible recommendation without SME input — it surfaces the case for expert review rather than guessing.

> **Outage Manager** — *No data means no confident recommendation. Who is the SME for this system, and how quickly can they assess?*  
> **Data Scientist** — *Watch how the zero-analog count drives confidence tier to `low_confidence`, which triggers the MONITOR branch in Stage G.*


In [ ]:
print('=== STAGE A — Unknown Component Scenario ===')
intake3 = r3['intake']
print(f"  Emergence type        : {intake3.get('emergence_type')}")
print(f"  Safety related        : {intake3.get('safety_related')}")
print(f"  Regulatory drivers    : {len(intake3.get('regulatory_drivers', []))}")
print(f"  Unknown abbr rate     : {intake3.get('unknown_abbreviation_rate', 0):.2%}")
print(f"  Data quality score    : {intake3.get('data_quality_score', 0):.2f}")
fig, axes = plot_stage_a_summary(intake3, title='Stage A — Unknown Component Scenario')
plt.show()


In [ ]:
# Stage C — empty KG driver → no chain links → causal_posture='insufficient_data'
fig, ax = plot_allen_timeline(r3['temporal'],
                               title='Stage C — Unknown Component (No History)')
plt.show()
print('Causal posture:', r3['temporal'].get('summary', {}).get('causal_posture'))


In [ ]:
# Stage D — zero analogs → low_confidence tier, no distribution to plot
fig, axes = plot_analog_distribution(r3['analogs'],
                                       title='Stage D — Unknown Component (Zero Analogs)')
plt.show()
rsm = r3['analogs'].get('retrieval_summary', {})
print(f"Analog count    : {rsm.get('analog_count', 0)}")
print(f"Confidence tier : {r3['analogs'].get('duration_distribution', {}).get('confidence_tier')}")


In [ ]:
# Stage E — non-critical, 48 h float available → no CP impact
fig, axes = plot_schedule_impact(r3['schedule'],
                                  title='Stage E — Unknown Component Schedule Impact')
plt.show()


In [ ]:
# Stage F — defer_to_post_outage blocked (safety_related=True)
# insert_now becomes primary but MONITOR fires before final recommendation
fig, ax = plot_option_risk_scores(r3['options'],
                                   title='Stage F — Unknown Component Option Risk Scores')
plt.show()


In [ ]:
# Stage G — MONITOR decision
print_recommendation_card(r3)
fig, ax = plot_recommendation_card(r3)
plt.show()


---
## 5 · Side-by-Side Comparison

The chart below directly compares the two scenarios across five key pipeline metrics.
This is useful for show-and-tell to illustrate how the same pipeline adapts
its recommendation based on input characteristics.

**What to highlight for managers:**
- CP drag and regulatory constraint are the two dominant factors
- Analogue count determines confidence tier, which shifts the risk balance in Stage F
- The pipeline reaches opposite conclusions (ESCALATE vs PROCEED) with full traceability


In [ ]:
fig, axes = plot_scenario_comparison(r1, r2, r3)
plt.show()


---
## 6 · Evidence Chain Traceability

Every recommendation must cite its sources — this is a **trust architecture requirement** (§8).

The evidence chain lists every data item that informed the decision:
historical work orders, condition reports, schedule float data, and temporal events.
An analyst can click into any item to see the raw source record.

**For managers:** This is the audit trail. If the recommendation is ever questioned,
the evidence chain shows exactly what data drove the conclusion and how confident
the system was in each source.

**For developers:** The evidence chain is assembled in Stage G's
`_build_evidence_chain()` method, which pulls from all upstream stage outputs.


In [ ]:
print('=== Scenario 1 Evidence Chain ===')
fig, ax = plot_evidence_chain(r1, title='Stage G — RCP Seal Evidence Chain')
plt.show()

print('\n=== Scenario 2 Evidence Chain ===')
fig, ax = plot_evidence_chain(r2, title='Stage G — Snubber Evidence Chain')
plt.show()

print('\n=== Scenario 3 Evidence Chain ===')
fig, ax = plot_evidence_chain(r3, title='Stage G — Unknown Component Evidence Chain')
plt.show()


---
## 7 · Trust Architecture Verification (§8 Requirements)

The `critical_analysis.md §8` trust architecture mandates that every recommendation
surface **four required fields** to enable analyst verification:

| Field | Purpose |
|-------|---------|
| (a) Outage IDs | Which historical outages contributed to the analogue pool |
| (b) Analog count | How many similar past activities were found |
| (c) Confidence tier | `data_supported` / `sme_informed` / `low_confidence` |
| (d) Rejection path | If the analyst rejects this recommendation, why and what was considered |

The cell below verifies that all four fields are present in both scenario outputs.


In [ ]:
# Verify that all four §8 trust-architecture fields are present in both outputs.
# This is a quick compliance check — in production, this would be a formal validator.
for label, r in [('Scenario 1 — RCP Seal', r1), ('Scenario 2 — Snubber', r2)]:
    rec   = r['recommendation']
    hist  = rec.get('history_summary', {})
    summ  = rec.get('executive_summary', {})
    rev   = rec.get('analyst_review', {})

    print(f'--- {label} ---')

    # (a) Outage IDs — the historical outages contributing analogues
    print(f'  (a) Outage IDs       : {hist.get("outage_ids", [])}')

    # (b) Analog count — how many similar past activities were retrieved
    print(f'  (b) Analog count     : {hist.get("analog_count", "?")}')

    # (c) Confidence tier — data_supported / sme_informed / low_confidence
    print(f'  (c) Confidence tier  : {summ.get("confidence_tier", "?")}')

    # (d) Rejection reason — None until an analyst explicitly rejects the recommendation;
    # we show a descriptive placeholder rather than bare 'None' for clarity
    rej = rev.get('rejection_reason')
    display = rej if rej else '(awaiting analyst action — not yet rejected)'
    print(f'  (d) Rejection reason : {display}')
    print()


---
## 8 · Raw Artifacts Inspector

Every stage output is a JSON-serialisable Python dictionary.
Change `STAGE_TO_INSPECT` below to explore any stage's full output structure.

**Available keys:** `intake`, `timeline`, `temporal`, `analogs`, `schedule`, `options`, `recommendation`

**For developers:** These raw artifacts are what would be stored to a database or
passed to downstream consumers (dashboard, notification service, audit log)
in a production deployment.


In [ ]:
import json

# Change these two values to inspect any stage's output for either scenario
STAGE_TO_INSPECT = 'recommendation'   # one of: intake, timeline, temporal, analogs,
                                      #         schedule, options, recommendation
SCENARIO         = r1                 # r1 = RCP Seal (ESCALATE)  |  r2 = Snubber (PROCEED)

# Pretty-print the selected artifact as JSON for easy inspection
artifact = SCENARIO.get(STAGE_TO_INSPECT, SCENARIO)
print(json.dumps(artifact, indent=2, default=str))
